In [91]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_recall_curve,log_loss,accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,matthews_corrcoef,roc_auc_score
import optuna
import gc
from scipy.stats import norm
from collections import Counter
from scipy.stats import norm
from functools import partial
from sklearn.model_selection import StratifiedKFold

In [92]:
# Bin bioactivity values into binary labels
def bio_cal(bioactivity):
    # Clip PKIS1 values because some entries are above 100 or below 0
    bioactivity = bioactivity.clip(0,100)
    # Binning rule: define fixed intervals and their corresponding labels
    bins = [-np.inf, 50, np.inf]  # Intervals used to classify the normalized bioactivity values
    labels = [0, 1]  # Binary labels for activity strength
    
    final_bioactivity = pd.cut(
    bioactivity, 
    bins=bins, 
    labels=labels, 
    include_lowest=True
    )
    return final_bioactivity

In [ ]:
band_range = '0.50_0.90'
band_range_2 = '50_90'
# import training data
warm_train_mol_fp = pd.read_csv(f"../dataset/spectrum_data/band_{band_range}/train_LSAE_latent_top95_features_2.csv",skiprows=1,header=None)
# import mol LSAE embedded features
warm_train = pd.read_csv(f"../dataset/spectrum_data/band_{band_range}/train.csv")
# import KLIFS seq from training data
warm_klifs = warm_train['KLIFS']
# import bioactivity classification from data
warm_bio = warm_train['unified_bio_label']

# mapping KLIFS to LSAE embedded KLIFS feature
KLIFS_sae_top = pd.read_csv("../feature_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv",skiprows=1, header=None).values
KLIFS_file = pd.read_csv("../feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv")

In [94]:
def klifs_map(df,KLIFS_file,KLIFS_latent): # df is the dataset to map; KLIFS_file contains all KLIFS entries; KLIFS_latent contains the corresponding reduced KLIFS features
    # Read KLIFS sequences from the input dataset
    read_klifs = df['KLIFS'].astype(str).str.strip()
    # Read KLIFS sequences from the KLIFS reference dataset
    KLIFS = KLIFS_file['KLIFS'].astype(str).str.strip()
    # Map each KLIFS sequence to its reduced latent representation
    keys = KLIFS
    values = KLIFS_latent
    KLIFS_dict = dict(zip(keys,values))
    # Perform the mapping
    kinase_result = [KLIFS_dict.get(x.lstrip(),None) for x in read_klifs]
    return kinase_result

In [95]:
# Concatenate molecular latent features and kinase latent features
warm_train_kinase_fp = pd.DataFrame(klifs_map(warm_train,KLIFS_file,KLIFS_sae_top))
warm_train_input_feature = pd.concat([warm_train_mol_fp,warm_train_kinase_fp],axis=1)

In [96]:
print(warm_train_mol_fp.shape)
print(warm_train_kinase_fp.shape)
print(warm_train_input_feature.shape)
print(warm_train_input_feature.shape)

(100000, 466)
(100000, 200)
(100000, 666)
(100000, 666)


In [97]:
# Delete intermediate variables to reduce memory usage
del warm_train_mol_fp,warm_train,warm_klifs
# Run garbage collection
gc.collect()

0

In [98]:
def dataset(input_feature, bioactivity):
    # Process feature labels
    input_feature.columns = range(input_feature.shape[1])
    print(input_feature.shape)
    # Safety check
    assert len(input_feature) == len(bioactivity), "The number of feature rows does not match the number of labels!"
    # Build the DMatrix object
    Xy = xgb.DMatrix(input_feature, label=bioactivity)
    return Xy,input_feature,bioactivity

In [99]:
Xy_train,X_train,y_train= dataset(warm_train_input_feature,warm_bio)

(100000, 666)


In [100]:
# set bedroc
def calculate_bedroc(y_true, y_score, bedroc_alpha):
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and extract the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Centered rank

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the theoretical limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize the score
    return (rie - rie_min) / (rie_max - rie_min)

In [101]:
# Calculate the enrichment factor
def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0,4.0, 5.0]):

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have the same length")
    
    N = len(y_true)
    n_active = np.sum(y_true == 1)
    
    if n_active == 0:
        print("Warning: no active compounds were found; returning EF = 0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        n_selected = max(1, int(N * percentage / 100))
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        results[f"EF_{percentage}%"] = ef_value
        print(f"EF_{percentage}%:")
        print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
        print(f"  - Active compounds among selected compounds: {n_active_in_selected}")
        print(f"  - Active-compound fraction: {n_active_in_selected/n_selected:.4f}")
        print(f"  - Overall active-compound fraction: {n_active/N:.4f}")
        print(f"  - EF value: {ef_value:.4f}")
        print()
    
    return results

In [102]:
def objective(trial, X, y, n_folds, bedroc_alpha):

    # ---------- 1. Search space ----------
    params = {
        "tree_method": "hist",
        "device": 'cuda',
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":     trial.suggest_int("max_depth", 5, 10),
        "subsample":     trial.suggest_float("subsample", 0.3, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),
        "lambda": trial.suggest_float("lambda", 1e-3, 1e3, log=True),
        "alpha":  trial.suggest_float("alpha", 1e-3, 300, log=True),
        "gamma":  trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "scale_pos_weight": 14,
    }

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    all_y_true = []
    all_y_pred = []
    best_iters = []

    for train_idx, valid_idx in skf.split(X, y):
        dtrain = Xy_train.slice(train_idx)
        dval = Xy_train.slice(valid_idx)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=10_000,
            evals=[(dval, "eval")],
            early_stopping_rounds=50,
            verbose_eval=False,
        )
        best_iters.append(model.best_iteration + 1)

        y_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))

        all_y_true.append(y[valid_idx])
        all_y_pred.append(y_pred)

    avg_best_iter = int(np.round(np.mean(best_iters)))
    trial.set_user_attr('avg_best_iter', avg_best_iter)

    y_true_concat = np.concatenate(all_y_true)
    y_pred_concat = np.concatenate(all_y_pred)

    bedroc = calculate_bedroc(y_true_concat, y_pred_concat, bedroc_alpha=bedroc_alpha)
    auc = roc_auc_score(y_true_concat, y_pred_concat)
    trial.set_user_attr("roc_auc", auc)
    trial.set_user_attr("bedroc", bedroc)
    
    print(f"[Validation] BEDROC alpha = {bedroc_alpha}")
    print(f"the best iteration number is {avg_best_iter}")
    return auc

In [103]:
# best hyperparameters and iteration number from full training pool model
best_hyperparameters= {
    'learning_rate': 0.015475107270785503, 
    'max_depth': 10, 
    'subsample': 0.8686837037964923, 
    'colsample_bytree': 0.6499639423346774, 
    'lambda': 691.5418676874641, 
    'alpha': 18.697404020910554, 
    'gamma': 0.007977823095946579, 
    'min_child_weight': 2}

# best_iter is not used here; max_iter and early stopping are used instead
best_iter = 9046
max_iter = 20000

early_stop_rounds = 50

In [104]:
# Load one pre-trained model for each fold from the existing model path
from pathlib import Path
n_folds = 5
fold_models = []

for fold_idx in range(n_folds):
    model_path = f"../model/KinSPEC_MCA_training_band_{band_range_2}_fold{fold_idx + 1}.json"
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Pre-trained model not found: {model_path}")

    model = xgb.Booster()
    model.load_model(model_path)
    fold_models.append(model)
    print(f"Fold {fold_idx + 1} model loaded from: {model_path}")


Fold 1 model loaded from: ../model/KinSPEC_MCA_training_band_50_90_fold1.json
Fold 2 model loaded from: ../model/KinSPEC_MCA_training_band_50_90_fold2.json
Fold 3 model loaded from: ../model/KinSPEC_MCA_training_band_50_90_fold3.json
Fold 4 model loaded from: ../model/KinSPEC_MCA_training_band_50_90_fold4.json
Fold 5 model loaded from: ../model/KinSPEC_MCA_training_band_50_90_fold5.json


In [105]:
del Xy_train,X_train,y_train,warm_train_input_feature,warm_bio
gc.collect()

14

In [106]:
# Define a function to test trained models using an ensemble setting
def run_the_model_ensemble(Xy, y, fold_models, kinase_name):
    
    # Generate predictions from each fold-specific model
    individual_preds = []
    for model in fold_models:
        pred = model.predict(Xy)
        individual_preds.append(pred)
    
    # Ensemble prediction by averaging fold predictions
    y_pred_prob = np.mean(individual_preds, axis=0)

    # Automatically select the optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    y_pred = (y_pred_prob > thresholds[optimal_idx])

    # Calculate common evaluation metrics
    auc = roc_auc_score(y, y_pred_prob)
    auc_var = np.var(y_pred_prob) / len(y_pred_prob)
    z = norm.ppf(1 - (1 - 0.95) / 2)
    lower_bound = auc - z * np.sqrt(auc_var)
    upper_bound = auc + z * np.sqrt(auc_var)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)
    logloss = log_loss(y, y_pred_prob)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0.0

    # Calculate confidence intervals by bootstrapping
    n_bootstraps = 1000
    bootstrapped_aucs = []
    y = np.array(y)
    y_pred_prob = np.array(y_pred_prob)

    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.choice(len(y), size=len(y), replace=True)
        auc_bootstrap = roc_auc_score(y[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(auc_bootstrap)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)
    print(f"Ensemble AUC: {auc:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Calculate BEDROC
    bedroc_score = calculate_bedroc(y, y_pred_prob, bedroc_alpha=80.5)
    ef_score = calculate_ef(y, y_pred_prob)
    
    # Print ensemble evaluation results
    print(f"\n========== Ensemble model evaluation results ==========")
    print(f"Best threshold: {thresholds[optimal_idx]:.4f}")
    print(f"AUC: {auc:.4f} (95% CI: {lower_bound:.3f}, {upper_bound:.3f})")
    print(f"bedroc: {bedroc_score:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(PPV): {precision:.4f}")
    print(f"Recall(Sensitivity): {recall:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV: {npv:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Log Loss: {logloss:.4f}")
    print(confusion_matrix(y, y_pred))
    print(f"True Negative = {tn}")
    print(f"True Positive = {tp}")
    print(f"False Negative = {fn}")
    print(f"False Positive = {fp}")
    print("Class distribution:", Counter(y))
    print("Positive-class fraction:", np.mean(y))
    
    # Calculate metrics for each fold-specific model
    print(f"\n========== Performance of each fold-specific model ==========")
    fold_aucs = []
    fold_bedrocs = []
    fold_accs = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_mccs = []
    fold_specificities = []
    fold_npvs = []
    fold_loglosses = []
    fold_efs = {f"EF_{p}%": [] for p in [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]}
    
    for i, pred in enumerate(individual_preds):
        ind_auc = roc_auc_score(y, pred)
        ind_bedroc = calculate_bedroc(y, pred, 80.5)
        
        ind_precisions, ind_recalls, ind_thresholds = precision_recall_curve(y, pred)
        ind_f1_scores = 2 * (ind_precisions * ind_recalls) / (ind_precisions + ind_recalls + 1e-10)
        ind_optimal_idx = np.argmax(ind_f1_scores)
        ind_y_pred = (pred > ind_thresholds[ind_optimal_idx])
        
        ind_acc = accuracy_score(y, ind_y_pred)
        ind_precision = precision_score(y, ind_y_pred)
        ind_recall = recall_score(y, ind_y_pred)
        ind_f1 = f1_score(y, ind_y_pred)
        ind_mcc = matthews_corrcoef(y, ind_y_pred)
        ind_logloss = log_loss(y, pred)
        ind_ef = calculate_ef(y, pred)
        
        ind_tn, ind_fp, ind_fn, ind_tp = confusion_matrix(y, ind_y_pred).ravel()
        ind_specificity = ind_tn / (ind_tn + ind_fp) if (ind_tn + ind_fp) != 0 else 0.0
        ind_npv = ind_tn / (ind_tn + ind_fn) if (ind_tn + ind_fn) != 0 else 0.0
        
        fold_aucs.append(ind_auc)
        fold_bedrocs.append(ind_bedroc)
        fold_accs.append(ind_acc)
        fold_precisions.append(ind_precision)
        fold_recalls.append(ind_recall)
        fold_f1s.append(ind_f1)
        fold_mccs.append(ind_mcc)
        fold_specificities.append(ind_specificity)
        fold_npvs.append(ind_npv)
        fold_loglosses.append(ind_logloss)
        for k, v in ind_ef.items():  
            fold_efs[k].append(v)
        
        print(f"Fold {i+1}: AUC={ind_auc:.4f}, BEDROC={ind_bedroc:.4f}, F1={ind_f1:.4f}, MCC={ind_mcc:.4f}")
    
    # Print mean and standard deviation
    print(f"\n========== Metric Mean +/- Std (across {len(fold_models)} folds) ==========")
    print(f"AUC:         {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}")
    print(f"BEDROC:      {np.mean(fold_bedrocs):.4f} +/- {np.std(fold_bedrocs):.4f}")
    print(f"Accuracy:    {np.mean(fold_accs):.4f} +/- {np.std(fold_accs):.4f}")
    print(f"Precision:   {np.mean(fold_precisions):.4f} +/- {np.std(fold_precisions):.4f}")
    print(f"Recall:      {np.mean(fold_recalls):.4f} +/- {np.std(fold_recalls):.4f}")
    print(f"F1 Score:    {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")
    print(f"Specificity: {np.mean(fold_specificities):.4f} +/- {np.std(fold_specificities):.4f}")
    print(f"NPV:         {np.mean(fold_npvs):.4f} +/- {np.std(fold_npvs):.4f}")
    print(f"MCC:         {np.mean(fold_mccs):.4f} +/- {np.std(fold_mccs):.4f}")
    print(f"Log Loss:    {np.mean(fold_loglosses):.4f} +/- {np.std(fold_loglosses):.4f}")
    for k, vals in fold_efs.items():   
        print(f"{k}:      {np.mean(vals):.4f} +/- {np.std(vals):.4f}")
        
    if kinase_name is not None:
        df_output = pd.DataFrame({
            "Best threshold": f"{thresholds[optimal_idx]:.4f}",
            "AUC": f"{auc:.4f}",
            "AUC 95% CI lower_bound": f"{lower_bound:.3f}",
            "AUC 95% CI upper_bound": f"{upper_bound:.3f}",
            "bedroc": f"{bedroc_score:.4f}",
            "Accuracy": f"{acc:.4f}",
            "Precision(PPV)": f"{precision:.4f}",
            "Recall(Sensitivity)": f"{recall:.4f}",
            "F1 score": f"{f1:.4f}",
            "Specificity": f"{specificity:.4f}",
            "NPV": f"{npv:.4f}",
            "MCC": f"{mcc:.4f}",
            "Log Loss": f"{logloss:.4f}",
            "True Negative": f"{tn}",
            "True Positive": f"{tp}",
            "False Negative": f"{fn}",
            "False Positive": f"{fp}"
        },index=[kinase_name])
        return bedroc_score,df_output,ef_score
    else:
        return bedroc_score

In [ ]:
vsds_mol_fp = pd.read_csv("../dataset/VSDS_truedecoy_LSAE_latent_top95_features.csv",skiprows=1,header=None)

vsds = pd.read_csv("../dataset/VSDS_truedecoy_integrated_data.csv")

vsds_klifs = vsds['KLIFS']

vsds_bio = vsds['molecule_label']

vsds_kinase_fp = pd.DataFrame(klifs_map(vsds,KLIFS_file,KLIFS_sae_top))
vsds_input_feature = pd.concat([vsds_mol_fp,vsds_kinase_fp],axis=1)

vsds_input_feature.columns = range(vsds_input_feature.shape[1])

print(vsds_mol_fp.shape)
print(vsds_kinase_fp.shape)
print(vsds_input_feature.shape)

vsds_Xy = xgb.DMatrix(vsds_input_feature,label=vsds_bio)

(49705, 466)
(49706, 200)
(49706, 666)


In [108]:
print('evaluation metrics on VSDS test set:')
vsds_test = run_the_model_ensemble(vsds_Xy, vsds_bio, fold_models, None)

evaluation metrics on VSDS test set:
Ensemble AUC: 0.655 (95% CI: [0.641, 0.671])
EF_0.5%:
  - Selected compounds: 248/49706 (0.5%)
  - Active compounds among selected compounds: 30
  - Active-compound fraction: 0.1210
  - Overall active-compound fraction: 0.0241
  - EF value: 5.0191

EF_1.0%:
  - Selected compounds: 497/49706 (1.0%)
  - Active compounds among selected compounds: 55
  - Active-compound fraction: 0.1107
  - Overall active-compound fraction: 0.0241
  - EF value: 4.5915

EF_2.0%:
  - Selected compounds: 994/49706 (2.0%)
  - Active compounds among selected compounds: 84
  - Active-compound fraction: 0.0845
  - Overall active-compound fraction: 0.0241
  - EF value: 3.5063

EF_3.0%:
  - Selected compounds: 1491/49706 (3.0%)
  - Active compounds among selected compounds: 112
  - Active-compound fraction: 0.0751
  - Overall active-compound fraction: 0.0241
  - EF value: 3.1167

EF_4.0%:
  - Selected compounds: 1988/49706 (4.0%)
  - Active compounds among selected compounds: 13